In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Imports and Global Paths

In [2]:
import re
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [6]:
# ===== PATH CONFIGURATION =====
BASE_DIR = "/content/drive/MyDrive/Assamese_Sentiment_Analysis/"

MODEL_DIR = BASE_DIR + "Models/Best_Model/muril/muril_finetuned/"
RESULTS_DIR = BASE_DIR + "Results/Predictions/"


In [7]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


# Load Best Model and Tokenizer

In [11]:
MODEL_NAME = MODEL_DIR

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

The tokenizer you are loading from '/content/drive/MyDrive/Assamese_Sentiment_Analysis/Models/Best_Model/muril/muril_finetuned/' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

# Assamese Negation, Contrast & Sarcasm Resources

In [46]:
# Assamese Negation Words
ASSAMESE_NEGATIONS = {
    "বিৰক্ত", "অসহ্য", "বেয়া", "কষ্ট", "সময় নষ্ট",
    "টোপনি", "শুইছিলোঁ"
}

# Contrast Markers
ASSAMESE_CONTRASTS = {
    "কিন্তু", "তথাপিও", "তথাপি", "বৰঞ্চ", "মাজতে", "যে", "কাৰণ"
}

# Sarcastic Cues
SARCASM_CUES = {
    "ইমান", "ভাল", "বৰ", "পকা", "প্ৰিয়", "পছন্দ", "নেকি?", "ধুনীয়া", "উৎকৃষ্ট"
}


# Linguistic Signal Detector

In [47]:
def detect_linguistic_signals(text):
    negation_found = any(n in text for n in ASSAMESE_NEGATIONS)
    contrast_found = any(c in text for c in ASSAMESE_CONTRASTS)
    sarcasm_found = any(p in text for p in SARCASM_CUES)

    return {
        "negation": negation_found,
        "contrast": contrast_found,
        "sarcasm_risk": sarcasm_found
    }


# Prediction with Explanation

In [52]:
def predict_with_explanation(text):
    signals = detect_linguistic_signals(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    base_label = "positive" if probs[1] > probs[0] else "negative"
    confidence = float(max(probs))

    final_label = base_label
    override_applied = False

    if (signals["negation"] and signals["contrast"] and signals["sarcasm_risk"]):
      final_label = "negative"
      override_applied = True

    return {
        "Text": text,
        "Base Prediction": base_label,
        "Final Prediction": final_label,
        "Confidence": round(confidence, 4),
        "Negation Detected": signals["negation"],
        "Contrast Detected": signals["contrast"],
        "Sarcasm Risk": signals["sarcasm_risk"],
        "Rule Override Applied": override_applied
    }


In [53]:
unseen_reviews = [
    "এইখন বৰ ভাল নাটক",
    "ভাল আছিল কিন্তু শেষটো একেবাৰে বেয়া",
    "মই এই ছবিখন ভাল পাইছিলো কাৰণ মই দুঘণ্টা ভালদৰে শুইছিলোঁ।",
    "এইটো মই কেতিয়াও নেদেখিম",
    "ঐ, কালি গুৱাহাটীৰ সেই নতুন কেফেখনলৈ গৈছিলোঁ। সঁচা ক’বলৈ গ’লে, Vibeটো একদম Next Level আছিল, কিন্তু পইচাটো মৰা। এটা কফিৰ দামত ঘৰত এসপ্তাহৰ চেনি-চাহপাত আহি যাব। ফ’টো উঠিবলৈ ভাল, কিন্তু পকেটৰ বাবে Not Recommended ভাই!",
    "আজিকালিৰ অসমীয়া মুভিবোৰত মিছাতে ইমান Melodrama কিয় দিয়ে বুজি নাপাওঁ। অলপ Practical হ’ব নোৱাৰে নেকি? ব্ৰেকআপ হ’লেই বৰষুণত তিতি কান্দিবই লাগিব নেকি? Cringe লাগি যায় কেতিয়াবা!",
    "যিহে জাম দিছে নহয়, ৰাস্তাতে বুঢ়া হৈ যাম যেন লাগিছে। অফিচ গৈ পোৱাৰ আগতে মোৰ Energy শেষ। চৰকাৰে ফ্লাইঅ’ভাৰ বনাইছে ঠিকেই, কিন্তু মানুহৰ ট্ৰেফিক চেন্সটো কতিয়া আহিব?"
]

results = []

for review in unseen_reviews:
    output = predict_with_explanation(review)
    results.append(output)



In [54]:
import pandas as pd

df_predictions = pd.DataFrame(results)

# Reorder columns for clarity
df_predictions = df_predictions[
    [
        "Text",
        "Base Prediction",
        "Final Prediction",
        "Confidence",
        "Negation Detected",
        "Contrast Detected",
        "Sarcasm Risk",
        "Rule Override Applied"
    ]
]

df_predictions


,Text,Base Prediction,Final Prediction,Confidence,Negation Detected,Contrast Detected,Sarcasm Risk,Rule Override Applied
0,এইখন বৰ ভাল নাটক,positive,positive,0.9948,False,False,True,False
1,ভাল আছিল কিন্তু শেষটো একেবাৰে বেয়া,positive,negative,0.7685,True,True,True,True
2,মই এই ছবিখন ভাল পাইছিলো কাৰণ মই দুঘণ্টা ভালদৰে...,positive,negative,0.9947,True,True,True,True
3,এইটো মই কেতিয়াও নেদেখিম,negative,negative,0.9461,False,False,False,False
4,"ঐ, কালি গুৱাহাটীৰ সেই নতুন কেফেখনলৈ গৈছিলোঁ। স...",positive,positive,0.6268,False,True,True,False
5,আজিকালিৰ অসমীয়া মুভিবোৰত মিছাতে ইমান Melodrama...,negative,negative,0.9834,False,False,True,False
6,"যিহে জাম দিছে নহয়, ৰাস্তাতে বুঢ়া হৈ যাম যেন লা...",negative,negative,0.9806,False,True,False,False


In [59]:
pd.set_option("display.max_colwidth", None)

def highlight_positive(val):
    return "color: yellow; font-weight: bold" if val == "positive" else ""

def highlight_negative(val):
    return "color: red; font-weight: bold" if val == "negative" else ""

styled_table = (
    df_predictions.style
    .applymap(highlight_positive, subset=["Final Prediction"])
    .applymap(highlight_negative, subset=["Final Prediction"])
    .format({"Confidence": "{:.3f}"})
    .set_properties(
        subset=["Text"],
        **{
            "white-space": "pre-wrap",
            "word-wrap": "break-word",
            "max-width": "600px"
        }
    )
)

styled_table


/tmp/ipython-input-3362661110.py:11: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_positive, subset=["Final Prediction"])
/tmp/ipython-input-3362661110.py:12: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_negative, subset=["Final Prediction"])


,Text,Base Prediction,Final Prediction,Confidence,Negation Detected,Contrast Detected,Sarcasm Risk,Rule Override Applied
0,এইখন বৰ ভাল নাটক,positive,positive,0.995,False,False,True,False
1,ভাল আছিল কিন্তু শেষটো একেবাৰে বেয়া,positive,negative,0.768,True,True,True,True
2,মই এই ছবিখন ভাল পাইছিলো কাৰণ মই দুঘণ্টা ভালদৰে শুইছিলোঁ।,positive,negative,0.995,True,True,True,True
3,এইটো মই কেতিয়াও নেদেখিম,negative,negative,0.946,False,False,False,False
4,"ঐ, কালি গুৱাহাটীৰ সেই নতুন কেফেখনলৈ গৈছিলোঁ। সঁচা ক’বলৈ গ’লে, Vibeটো একদম Next Level আছিল, কিন্তু পইচাটো মৰা। এটা কফিৰ দামত ঘৰত এসপ্তাহৰ চেনি-চাহপাত আহি যাব। ফ’টো উঠিবলৈ ভাল, কিন্তু পকেটৰ বাবে Not Recommended ভাই!",positive,positive,0.627,False,True,True,False
5,আজিকালিৰ অসমীয়া মুভিবোৰত মিছাতে ইমান Melodrama কিয় দিয়ে বুজি নাপাওঁ। অলপ Practical হ’ব নোৱাৰে নেকি? ব্ৰেকআপ হ’লেই বৰষুণত তিতি কান্দিবই লাগিব নেকি? Cringe লাগি যায় কেতিয়াবা!,negative,negative,0.983,False,False,True,False
6,"যিহে জাম দিছে নহয়, ৰাস্তাতে বুঢ়া হৈ যাম যেন লাগিছে। অফিচ গৈ পোৱাৰ আগতে মোৰ Energy শেষ। চৰকাৰে ফ্লাইঅ’ভাৰ বনাইছে ঠিকেই, কিন্তু মানুহৰ ট্ৰেফিক চেন্সটো কতিয়া আহিব?",negative,negative,0.981,False,True,False,False


# Save Table

In [60]:
SAVE_PATH = RESULTS_DIR

df_predictions.to_csv(
    SAVE_PATH + "Unseen_Predictions_With_Linguistic_Awareness.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Prediction table saved successfully.")


Prediction table saved successfully.
